NoteBook 03

This notebook builds an advanced recommendation system using Sentence-BERT (SBERT) for semantic understanding. Unlike TF-IDF, it captures contextual meaning in text.

🔹 Key Steps:
    1.Load data
    2.Encode (BATCHED)
    3.Recommend
    4.Save embeddings

🔹 Purpose:
    Improve recommendation quality
    Capture meaning beyond keywords

1. Load

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

resume_df = pd.read_csv('../data/processed/processed_resume.csv')
jobs_df   = pd.read_csv('../data/processed/processed_jobs.csv')

2. Encode (BATCHED)

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

job_emb = model.encode(
    jobs_df['text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

resume_emb = model.encode(
    resume_df['text'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

3. Recommend

In [3]:
sim_matrix = cosine_similarity(resume_emb, job_emb)

def recommend(idx, top_n=5):
    scores = sim_matrix[idx]
    top_idx = scores.argsort()[::-1][:top_n]
    return jobs_df.iloc[top_idx][['title','location']]

4. Save embeddings

In [4]:
import numpy as np

np.save('../models/resume_emb.npy', resume_emb)
np.save('../models/job_emb.npy', job_emb)